In [3]:
import numpy as np
from numpy.linalg import norm
from math import pi
import matplotlib.pyplot as plt
from scipy.optimize import newton_krylov
from scipy.sparse.linalg import LinearOperator, gmres

# Set up spatial grid (N points on [0,1), periodic)
N = 1024 
X = np.linspace(0.0, 1.0, N, endpoint=False)
dx = 1.0 / N

# Residual function F(U) for the nonlinear system
def residual(U, eta):

    U = U.reshape(-1) 
    # Second derivative U'' using central differences
    U_xx = (np.roll(U, -1) - 2.0 * U + np.roll(U, 1)) / (dx**2)
    
    return -(1.0/eta**2) * U_xx + 4*pi*np.sqrt(2) * np.sin(2*pi*(X + np.sqrt(2)*U))

# Jacobian-vector product J(U)*v
def jacobian_action(v, U, eta):

    # Linear part: 
    v_xx = (np.roll(v, -1) - 2.0 * v + np.roll(v, 1)) / (dx**2)
    term_linear = -(1.0/eta**2) * v_xx
    # Nonlinear part: 
    term_nonlinear = 16 * (pi**2) * np.cos(2*pi*(X + np.sqrt(2)*U)) * v

    return term_linear + term_nonlinear

# Preconditioner: invert L = - (1/eta^2) d^2/dx^2 on the space of mean-zero functions.
def apply_linear_precond(r, eta):

    r_fft = np.fft.rfft(r) 
    k_vals = np.fft.rfftfreq(N, d=dx) 
    Y_fft = np.zeros_like(r_fft, dtype=complex)
    Y_fft[0] = 0.0  # set to 0 to enforce mean-zero solution.
    # Inverse for k>0
    for ik in range(1, len(r_fft)):
        k = k_vals[ik]
        Y_fft[ik] = (eta**2) * r_fft[ik] / ((2*pi * k)**2)
    y = np.fft.irfft(Y_fft, n=N)  # inverse FFT to real space
    return y.real

# Construct LinearOperator objects for Jacobian and preconditioner (for GMRES)
def make_J_linop(U, eta):
    """Return a LinearOperator representing the Jacobian J(U)."""
    def matvec(v):
        return jacobian_action(v, U, eta)
    return LinearOperator((N, N), matvec=matvec, dtype=float)

def make_precond_linop(eta):
    """Return a LinearOperator for the preconditioner M approx ≈ J^{-1} using linear operator inverse."""
    def matvec(r):
        # Ensure mean of r is zero
        r = r - np.mean(r)
        return apply_linear_precond(r, eta)
    return LinearOperator((N, N), matvec=matvec, dtype=float)

# Newton-Krylov solver with inexact Newton and line search
def solve_nonlinear(U_init, eta, tol=1e-8):
    """Solve F(U)=0 for given eta using Newton-Krylov. U_init is initial guess array."""
    U = U_init.copy()
    prev_F_norm = np.inf  
    # Pseudo-transient continuation: perform a few explicit or implicit steps to improve initial guess
    dt = 0.1  # initial pseudo-time step
    for step in range(10):
        F = residual(U, eta)
        dU = apply_linear_precond(-F * dt, eta)  # one step of backward Euler using linear inverse
        U += dU
        # Check if the solution is improving
        if norm(residual(U, eta), np.inf) < 0.5 * norm(F, np.inf):
            # solution improving, can increase dt to speed up
            dt = min(dt * 1.5, 1.0)
        else:
            # if not improving much, reduce dt for stability
            dt = dt * 0.5
    
    # Enforce mean-zero on initial guess
    U = U - np.mean(U)
    F = residual(U, eta)
    F_norm = norm(F, np.inf)
    # Newton iterations
    max_newton = 500
    inner_tol = 1e-2  # initial Krylov tolerance
    for k in range(max_newton):
        # Check convergence
        F_norm = norm(F, np.inf)
        if F_norm < tol:
            break
        # Set up Jacobian LinearOperator at current U and preconditioner
        J_op = make_J_linop(U, eta)
        M_op = make_precond_linop(eta)
        # Determine adaptive forcing term for inexact Newton (Eisenstat-Walker strategy):
        if k == 0:
            inner_tol = 1e-2
        else:
            # Adaptive update: e.g., η_k = min(0.1, (||F_new||/||F_old||)^0.5 )
            inner_tol = min(0.1, np.sqrt(norm(residual(U, eta), np.inf) / max(prev_F_norm, 1e-12)))
        prev_F_norm = F_norm
        # Solve J * dU = -F using GMRES (Jacobian-Free via J_op.matvec) with preconditioning
        dU, info = gmres(J_op, -F, M=M_op, tol=inner_tol, maxiter=100)
        if info != 0:
            print(f"Warning: GMRES did not fully converge (info={info}) at Newton iter {k}")
        # Trust-region / line search: check if full step decreases residual; if not, backtrack
        new_U = U + dU
        new_F = residual(new_U, eta)
        if norm(new_F, np.inf) > 0.9 * F_norm:
            # If insufficient decrease, backtrack
            alpha = 1.0
            while alpha > 1e-4:
                alpha *= 0.5
                new_U = U + alpha * dU
                new_F = residual(new_U, eta)
                if norm(new_F, np.inf) < F_norm:
                    # Accept the step if improvement is seen
                    break
            # If no improvement even for tiny step, abort (unlikely if proper preconditioning)
        # Update U and residual for next iteration
        U, F = new_U, new_F
        # Enforce mean-zero constraint after each update (remove any drift in the null space)
        U = U - np.mean(U)
        F = residual(U, eta)
    else:
        print("Warning: Newton iterations did not converge within max_newton")
    return U

# Solve for multiple eta values using continuation
etas = [4, 1.0, 0.3]  # parameter values to solve
solutions = []
U_guess = np.zeros(N)  # start with zero initial guess for the first parameter
for eta in etas:
    print(f"Solving for eta = {eta}...")
    U_sol = solve_nonlinear(U_guess, eta)
    # Make sure solution has zero mean
    U_sol -= np.mean(U_sol)
    solutions.append(U_sol.copy())
    # Use current solution as initial guess for next eta
    U_guess = U_sol

# Plot the solutions for each eta
plt.figure(figsize=(8,5))
for U_sol, eta in zip(solutions, etas):
    plt.plot(X, U_sol, label=f"$\eta = {eta}$")
plt.axhline(0, color='gray', linewidth=0.8)
plt.xlabel("X")
plt.ylabel("U(X)")
plt.title("Periodic Solution U(X) for 1D Moiré System at Various $\eta$")
plt.legend()
plt.grid(True)
plt.show()


Solving for eta = 4...
Solving for eta = 1.0...


KeyboardInterrupt: 